# Faz 9 — XGBoost'un Canlı Pipeline'a Entegrasyonu

`2.ipynb`'de kaydedilen XGBoost modelini (`grid_xgb_model.json`) yükler, gerçek son 24+ saatlik veriden lag özelliklerini hesaplar, tahmin üretir, ve **LSTM'in Faz 5'iyle birebir aynı** RAG→Agent→n8n zincirini çalıştırır — tek fark tahminin kaynağı. Güven aralığı, LSTM'in RMSPE'si (0,0273) değil, **XGBoost'un kendi RMSPE'siyle** (0,0097) hesaplanır — bu, canlı sistemde ~3× daha dar bir aralık üretir (bkz. sunum, Güven Aralığı Karşılaştırması slaydı).

In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np
import json, os, sys, requests
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langsmith import traceable

sys.path.append(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje\notepads")
from rag_retrieval import retrieve_context, build_rag_query
from prompt_v1 import SYSTEM_PROMPT

load_dotenv(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje\notepads\.env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# --- Adim 1: XGBoost modelini yukleyin ---
xgb_model = xgb.XGBRegressor()
xgb_model.load_model(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje\agent\grid_xgb_model.json")

# --- Adim 2: Gercek veriyi yukleyip lag ozelliklerini hesaplayin ---
df = pd.read_csv(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje\data\aep_with_features.csv", parse_dates=["Datetime"])
df = df.sort_values("Datetime").reset_index(drop=True)

xgb_features = ["hour", "day_of_week", "month", "is_weekend", "is_holiday",
                 "avg_temp_f", "HDD", "CDD",
                 "lag_1", "lag_2", "lag_3", "lag_24", "lag_168", "rolling_24h_mean"]

last_known = df.iloc[-1]
next_hour_dt = last_known["Datetime"] + pd.Timedelta(hours=1)

row = {
    "hour": next_hour_dt.hour, "day_of_week": next_hour_dt.dayofweek, "month": next_hour_dt.month,
    "is_weekend": next_hour_dt.dayofweek >= 5, "is_holiday": last_known["is_holiday"],
    "avg_temp_f": last_known["avg_temp_f"], "HDD": last_known["HDD"], "CDD": last_known["CDD"],
    "lag_1": df["AEP_MW"].iloc[-1], "lag_2": df["AEP_MW"].iloc[-2], "lag_3": df["AEP_MW"].iloc[-3],
    "lag_24": df["AEP_MW"].iloc[-24], "lag_168": df["AEP_MW"].iloc[-168],
    "rolling_24h_mean": df["AEP_MW"].iloc[-24:].mean(),
}
X_next = pd.DataFrame([row])[xgb_features]

# --- Adim 3: Tahmin uretin ---
forecast_mw = xgb_model.predict(X_next)[0]

# --- Adim 4: Guven araligi - XGBoost'un KENDI RMSPE'siyle ---
xgb_rmspe = 0.009716  # 2.ipynb'nin dogrulanmis sonucu
confidence_margin = forecast_mw * xgb_rmspe
confidence_interval = [round(forecast_mw - confidence_margin), round(forecast_mw + confidence_margin)]

# --- Adim 5: grid_status - LSTM'deki ile AYNI IRM mantigi ---
np.random.seed(42)
recent_peak_mw = df["AEP_MW"].tail(24*30).max()
available_capacity_mw = round(recent_peak_mw * 1.20 * np.random.uniform(0.97, 1.03))
hr = next_hour_dt.hour
time_period = "day" if 7 <= hr < 18 else ("peak" if 18 <= hr < 23 else "night")

demand_forecast = {"region": "AEP", "forecast_mw": round(forecast_mw), "confidence_interval": confidence_interval}
grid_status = {"available_capacity_mw": available_capacity_mw, "time_period": time_period,
               "capacity_source": "simulated_from_PJM_2027-2028_IRM_20pct"}

print("XGBoost tahmini:", demand_forecast, grid_status)

# --- Adim 6: RAG + Agent - LSTM'deki ile BIREBIR AYNI ---
rag_query = build_rag_query(forecast_mw, available_capacity_mw, time_period)
retrieved_context = retrieve_context(rag_query)
content = f"""<demand_forecast>
{json.dumps(demand_forecast)}
</demand_forecast>
<grid_status>
{json.dumps(grid_status)}
</grid_status>
<official_procedure_context>
{retrieved_context}
</official_procedure_context>"""

@traceable(name="grid_agent_decision_with_rag", metadata={"forecast_model": "XGBoost"})
def get_agent_decision(content, system_prompt):
    response = client.models.generate_content(
        model="gemini-3.7-flash", contents=content,
        config=types.GenerateContentConfig(system_instruction=system_prompt, temperature=0.2,
                                            response_mime_type="application/json"),
    )
    return response.text

decision_json = json.loads(get_agent_decision(content, SYSTEM_PROMPT))
decision_json["region"] = "AEP"
print("Agent karari (XGBoost tabanli):", decision_json)

# --- Adim 7: n8n'e gonder - AYNI webhook ---
result = requests.post("http://localhost:5678/webhook/grid-forecast", json=decision_json)
print(result.status_code, result.text)

XGBoost tahmini: {'region': 'AEP', 'forecast_mw': 13681, 'confidence_interval': [13549, 13814]} {'available_capacity_mw': 25570, 'time_period': 'night', 'capacity_source': 'simulated_from_PJM_2027-2028_IRM_20pct'}


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Agent karari (XGBoost tabanli): {'reasoning_steps': 'Forecasted demand of 13,681 MW is well within the available capacity of 25,570 MW during the low-demand night period. The substantial capacity surplus easily satisfies standard operating margins and reserve requirements.', 'status': 'normal', 'decision': 'normal_operation', 'justification': 'Grid capacity significantly exceeds projected demand, comfortably maintaining PJM Primary and Secondary Reserve Requirements under standard operating procedures.', 'uncertainty': 'low', 'confidence_note': 'Forecast confidence interval is tight (13,549–13,814 MW) and all necessary grid metrics are provided without contradictions.', 'region': 'AEP'}
200 {"message":"Workflow was started"}
